In [0]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier

# Prepare label and features
df = spark.table("workspace.gold.products")
df = df.withColumn("label", (df["purchases"] > 0).cast("double"))

# Define models
lr = LogisticRegression(featuresCol="features", labelCol="label")
rf = RandomForestClassifier(featuresCol="features", labelCol="label")
gbt = GBTClassifier(featuresCol="features", labelCol="label")
models = [("LogisticRegression", lr), ("RandomForest", rf), ("GBT", gbt)]

In [0]:
import mlflow
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

assembler = VectorAssembler(
    inputCols=["views", "purchases", "revenue", "conversion_rate"],
    outputCol="features",
    handleInvalid="skip"
)
train, test = df.randomSplit([0.7, 0.3], seed=42)

metrics = []
for name, model in models:
    with mlflow.start_run(run_name=name):
        pipeline = Pipeline(stages=[assembler, model])
        fitted = pipeline.fit(train)
        predictions = fitted.transform(test)
        accuracy = predictions.filter(predictions.label == predictions.prediction).count() / predictions.count()
        mlflow.log_metric("accuracy", accuracy)
        metrics.append((name, accuracy))

In [0]:
# Example: Build pipeline for best model (can be reused for all)
best_model_name, _ = max(metrics, key=lambda x: x[1])
best_model = dict(models)[best_model_name]
pipeline = Pipeline(stages=[assembler, best_model])
fitted_pipeline = pipeline.fit(train)

In [0]:
print(f"Best model: {best_model_name}")

Best model: LogisticRegression


In [0]:
mlflow.end_run()